其实之前在 `code\zero_gpt\bigram.py`代码中，
+ 即`7_从零开始构建GPT.md`中，`code\zero_gpt\7_start_GPT.ipynb`文件里，
+ 直接采用了最简单的分词办法，就是每个字符就是一个词表里的元素

但是更现代的分词器一般不是`character-level`的，而是`chunk-level`的，最常见的就是`BPE`(Byte-Pair Encode)算法
+ `Language Models are unsupervised multitask learners`这个文章，即:`GPT-2`中使用的就是`BPE`算法
+ [Llama 2: OpenFoundation and Fine-Tuned Chat Model](https://arxiv.org/pdf/2307.09288)

# Tokenization :(

## Unicode编码

`ord`函数接受单个的`Unicode code point`, 并返回对应的整数值

In [3]:
ord("H"), ord("✅"), ord("黄")

(72, 9989, 40644)

In [9]:
unicode_rs = [ord(x)for x in "안녕하세요 👋 (hello in Korean!)"]
print(len(unicode_rs), '\n', unicode_rs)

26 
 [50504, 45397, 54616, 49464, 50836, 32, 128075, 32, 40, 104, 101, 108, 108, 111, 32, 105, 110, 32, 75, 111, 114, 101, 97, 110, 33, 41]


如果只是单纯考虑把字符串转为正数，其实`Unicode`编码就完全可以实现这个功能，但是根据[Unicode](https://en.wikipedia.org/wiki/Unicode)可知：
1. Unicode的字符量很大，截止 version 18，包含`172,808`个字符以及175种脚本类型
2. Unicode本身在不停的演进变化，并不能作为一种稳定的字符表示方案

因此，人们没有选择直接使用`Unicode`这种方案，而是转向了编码方案，根据[Unicode](https://en.wikipedia.org/wiki/Unicode)
```
The Unicode Standard itself defines three encodings: UTF-8, UTF-16,[a] and UTF-32, 
```

[UTF-8](https://en.wikipedia.org/wiki/UTF-8)是目前最常用的方案
+ 我们最需要关注的一点是： `uft-8`会把上面的`Unicode code point`转为`byte stream`, 其中转换的字节流在1~4个字节长度，即： 这是一种**变长编码**
+ 相比于`UTF-16`,`UTF-32`, `UTF-8`有一个最显著的优点就是完全向后兼容`ASCII`编码
+ 关于编码方案，可以看看:
  + [A Programmer’s Introduction to Unicode](https://www.reedbeta.com/blog/programmers-intro-to-unicode/)
  + 上个文章的参考链接，这篇是关于为什么`UTF-8`可以广泛应用的： [UTF-8 Everywhere](https://utf8everywhere.org/)

In [6]:
"안녕하세요 👋 (hello in Korean!)".encode("utf-8")

b'\xec\x95\x88\xeb\x85\x95\xed\x95\x98\xec\x84\xb8\xec\x9a\x94 \xf0\x9f\x91\x8b (hello in Korean!)'

这其实是用`utf-8`编码后的原始字节流，下面是直接转为列表后显示的数字列表

In [10]:
uft_rs = list("안녕하세요 👋 (hello in Korean!)".encode("utf-8"))
print(len(uft_rs), '\n', uft_rs)

39 
 [236, 149, 136, 235, 133, 149, 237, 149, 152, 236, 132, 184, 236, 154, 148, 32, 240, 159, 145, 139, 32, 40, 104, 101, 108, 108, 111, 32, 105, 110, 32, 75, 111, 114, 101, 97, 110, 33, 41]


In [11]:
uft_rs = list("안녕하세요 👋 (hello in Korean!)".encode("utf-16"))
print(len(uft_rs), '\n', uft_rs)

56 
 [255, 254, 72, 197, 85, 177, 88, 213, 56, 193, 148, 198, 32, 0, 61, 216, 75, 220, 32, 0, 40, 0, 104, 0, 101, 0, 108, 0, 108, 0, 111, 0, 32, 0, 105, 0, 110, 0, 32, 0, 75, 0, 111, 0, 114, 0, 101, 0, 97, 0, 110, 0, 33, 0, 41, 0]


同理，也可以看看`uft-16`的编码结果，这里可以看到，由于`utf-16`用了更多的字节数编码，所以这里会出现大量的0字节

尤其是最后几个`hello`的ASCII字符，很明显在`utf-8`编码的时候，这些字符都是单个字节表示的，而`utf-16`出现了大量的`数字+0`的组合